# 1.Tagesschau

## 1.1 Scraper

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re

# Url of the website
url = 'https://www.tagesschau.de'

# Make a request to the website
r = requests.get(url)
r_html = r.text

# Create a BeautifulSoup object and specify the parser
soup = BeautifulSoup(r_html, 'html.parser')

# Find all article links on the page
potential_article_links = [a['href'] for a in soup.find_all('a', href=True) if a['href'].startswith('/') and len(a['href']) > 1]

# Filter out non-news and clean up the links, only keep those end with '.html' and don't contain 'wetter' or 'multimedia'
potential_article_links = ['https://www.tagesschau.de' + link for link in potential_article_links if link.endswith('.html') and 'wetter' not in link and 'multimedia' not in link]

# Function to remove links to other articles
def remove_date_and_following_lines(text):
    lines = text.split('\n')
    i = 0
    while i < len(lines):
        if re.fullmatch(r'\d{2}\.\d{2}\.\d{4}', lines[i].strip()):  # if the line matches the pattern
            del lines[i:i+3]  # remove this line and the next two lines
            continue  # skip the increment to stay at the current index
            #print(f"\n\n{i}\n\n")
        i += 1  # move to the next line
    return '\n'.join(lines)

datetime_list  = []
title_list  = []
article_list  = []
title_article_list  = []

# For each potential article link, make a request to the page and check if it's an article page
for i, potential_article_link in enumerate(potential_article_links):
    r = requests.get(potential_article_link)
    soup = BeautifulSoup(r.text, 'html.parser')
    
    # Check if it's an article page by looking for the <article> tag
    article_body = soup.find('article')
    
    if article_body:
        # Extract the title of the article
        title = soup.find('title').get_text()
        
        # Remove "| tagesschau.de" from the title
        title = title.replace(" | tagesschau.de", "")
        #print(f'{title}\n')
        
        paragraphs = article_body.find_all('p')
        article_content = ' '.join([p.get_text() for p in paragraphs])

        # Call function to remove links to other articles
        article_content = remove_date_and_following_lines(article_content)
        
        # Remove lines containing only a single word
        article_content = re.sub(r'^\s*\b\w+\b\s*$', '', article_content, flags=re.MULTILINE)
        
        # Remove lines containing a sequence of more than 5 whitespaces
        article_content = re.sub(r'^.*\s{6,}.*$', '', article_content, flags=re.MULTILINE)
        
        # Remove all line breaks
        article_content = re.sub(r'\n', ' ', article_content)
        
        # Remove double whitespaces
        article_content = re.sub(r'\s{2,}', ' ', article_content)
        
        # Trim the text from 'Dieses Thema im Programm' and whitespaces
        article_content = re.sub(r'Dieses Thema im Programm.*$', '', article_content, flags=re.DOTALL)
        article_content = article_content.strip()
        
        # Extract timestamp and remove from article_content
        match = re.search(r'Stand: (\d{2}\.\d{2}\.\d{4} \d{2}:\d{2}) Uhr', article_content)
        if match:
            dt_str = match.group(1)  # The first group is the date and time string
        else:
            dt_str = "N_A"
        article_content = re.sub(r'Stand: (\d{2}\.\d{2}\.\d{4} \d{2}:\d{2}) Uhr', '', article_content)
        
        title_list.append(title)
        article_list.append(article_content)
        title_article_list.append(title + ' ' + article_content)
        datetime_list.append(dt_str)
        
    # Sleep for a while to prevent overloading the server with requests
    time.sleep(1)

print(len(title_list))
print(len(article_list))
print(len(datetime_list))
print(len(title_article_list))
print(title_article_list)


## 1.2 Dataframe

In [58]:
# Set option to display more content in cells
pd.set_option('display.max_colwidth', 500)

In [167]:
# Create df from lists with scraped content

import pandas as pd

# Create a DataFrame:
tagesschau_df = pd.DataFrame({
    'title': title_list,
    'datetime': datetime_list,
    'article': article_list,
    'title_article': title_article_list
})

tagesschau_df['source'] = 'tagesschau'

tagesschau_df.head(2)

,title,datetime,article,title_article,source
0,"Digitalpolitik der Ampel: ""Wir können ja nur besser werden""",07.06.2023 09:17,"Unambitioniert und holprig - die Digitalpolitik der Ampelregierung fällt bei Fachleuten durch. Auf der re:publica sucht man nach Erklärungen - und was sagt eigentlich Minister Wissing dazu? Ohne Termin zu einem neuen Ausweis - und das in Berlin. Das Innenministerium macht es möglich - bei der re:publica. Allerdings nur, wenn man dafür ganz analog vor Ort ist. Den Ausweis im Netz beantragen, das geht auch im ""Pop-up Bürgeramt"" bei der Digitalkonferenz nicht. Für viele bei der Konferenz ist d...","Digitalpolitik der Ampel: ""Wir können ja nur besser werden"" Unambitioniert und holprig - die Digitalpolitik der Ampelregierung fällt bei Fachleuten durch. Auf der re:publica sucht man nach Erklärungen - und was sagt eigentlich Minister Wissing dazu? Ohne Termin zu einem neuen Ausweis - und das in Berlin. Das Innenministerium macht es möglich - bei der re:publica. Allerdings nur, wenn man dafür ganz analog vor Ort ist. Den Ausweis im Netz beantragen, das geht auch im ""Pop-up Bürgeramt"" bei d...",tagesschau
1,Nach Dammbruch: Verheerende Folgen für Menschen und Natur,07.06.2023 07:43,"Zehntausende Menschen sind laut Kiew nach der Explosion des Kachowka-Staudamms von den Fluten bedroht. Auch für die Landwirtschaft sind die Folgen gravierend: 10.000 Hektar Land könnten zu Wüsten werden, warnt das Agrarministerium. Etwa 42.000 Menschen sind ukrainischen Angaben zufolge nach der Zerstörung des Kachowka-Staudamms am Dnipro im Süden des Landes von Überschwemmungen bedroht. Auch der UN-Nothilfekoordinator Martin Griffiths erklärte vor dem Sicherheitsrat, dass der Dammbruch ""sch...","Nach Dammbruch: Verheerende Folgen für Menschen und Natur Zehntausende Menschen sind laut Kiew nach der Explosion des Kachowka-Staudamms von den Fluten bedroht. Auch für die Landwirtschaft sind die Folgen gravierend: 10.000 Hektar Land könnten zu Wüsten werden, warnt das Agrarministerium. Etwa 42.000 Menschen sind ukrainischen Angaben zufolge nach der Zerstörung des Kachowka-Staudamms am Dnipro im Süden des Landes von Überschwemmungen bedroht. Auch der UN-Nothilfekoordinator Martin Griffit...",tagesschau


## 1.3 CSV file

In [140]:
# Save df from 5 June to csv -> done
tagesschau_df.to_csv('tagesschau_5june.csv', index=False)

In [156]:
# Save df from 6 June to csv -> done
tagesschau_df.to_csv('tagesschau_6june.csv', index=False)

In [168]:
# Save df from 7 June to csv (not yet applied)
tagesschau_df.to_csv('tagesschau_7june.csv', index=False)

# 2. Berliner Kurier

## 2.1 Scraper

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

def get_article_body_berliner_kurier(url):
    article_html = requests.get(url)
    article_soup = BeautifulSoup(article_html.text, 'html.parser')

    #title = article_soup.find('h1', class_="atc-HeadlineText").text.strip()
    title = article_soup.find('h1', class_="a-storyhead").text.strip()
    
    # Find the main article content
    #article_body = article_soup.find('div', class_="RichText RichText--iconLinks lg:w-8/12 RichText--lastPmb0").text.strip()
    
    article_body = article_soup.find_all('p', class_="a-paragraph")
    #article_body = article_soup.find('div', class_="o-article")
    
    return title, article_body

def get_articles_berliner_kurier(main_url):
    # Send GET request
    res = requests.get(main_url)

    # Parse HTML content
    soup = BeautifulSoup(res.text, 'html.parser')
    
    # Find all links
    links = soup.find_all('a', href=True)
    
    # Filter for article links
    article_links = [main_url + link['href'] for link in links if '-li.' in link['href']]
    
    # Create lists for articles and titles
    article_list = []
    title_list = []
    title_article_list = []

    # Go through all the article links and get the article body
    for link in article_links:
        #print(link)
        try:
            title, body = get_article_body_berliner_kurier(link)
            
            
            if body is not None:
                body = str(body)
                body = re.sub('<.*?>', '', body)
                
                # Remove lines containing only a single word
                body = re.sub(r'^\s*\b\w+\b\s*$', '', body, flags=re.MULTILINE)
        
                # Remove all line breaks
                body = re.sub(r'\n', ' ', body)
        
                # Remove double whitespaces
                body = re.sub(r'\s{2,}', ' ', body)
                
                # Remove [ at the beginning and ] at the end of the line
                body = re.sub(r'^\[|\]$', '', body)
                
                # Remove links to other articles
                body = re.sub(r', Lesen Sie auch.*?&gt;&gt;', '', body)
            
            article_list.append(body)
            title_list.append(title)
            title_article_list.append(title + ' ' + body)
               
        except Exception as e:
            print(f"Error getting article body from {link}: {e}")
    return article_list, title_list, title_article_list

# Usage:
main_url = "https://www.berliner-kurier.de"
articles, titles, titles_articles = get_articles_berliner_kurier(main_url)

print(len(title_list))
print(len(article_list))
print(len(title_article_list))

#print(articles)

for title, article in zip(titles, articles):
    #article = re.sub('<.*?>', '', article)
    print("-"*80)
    print(title)
    print(article)

    


## 2.2 Dataframe

In [170]:
# Create df from lists with scraped content

import pandas as pd

# Create a DataFrame:
kurier_df = pd.DataFrame({
    'title': titles,
    'datetime': "N_A",
    'article': articles,
    'title_article': titles_articles
})

kurier_df['source'] = 'berliner_kurier'

kurier_df.head(2)


,title,datetime,article,title_article,source
0,"Süß, saftig, wenig Kerne: Daran erkennen Sie die perfekte Wassermelone",N_A,"Sommerzeit ist Melonen-Zeit. Kaum ein anderes Obst ist in der warmen Jahreszeit so beliebt, wie köstliche Wassermelonen. Kein Wunder, erfrischt es doch ungemein, ist saftig süß und auf Grund des hohen Wasseranteils trotzdem arm an Kalorien. Aber: Wie erkennt man eigentlich eine Melone, die besonders süß und saftig ist? Hier kommen die ultimativen Tipps. , Geschlecht: Kein Witz: Auch Wassermelonen haben ein Geschlecht. Und – sorry Jungs – die Mädels schmecken einfach besser. Denn männliche Wa...","Süß, saftig, wenig Kerne: Daran erkennen Sie die perfekte Wassermelone Sommerzeit ist Melonen-Zeit. Kaum ein anderes Obst ist in der warmen Jahreszeit so beliebt, wie köstliche Wassermelonen. Kein Wunder, erfrischt es doch ungemein, ist saftig süß und auf Grund des hohen Wasseranteils trotzdem arm an Kalorien. Aber: Wie erkennt man eigentlich eine Melone, die besonders süß und saftig ist? Hier kommen die ultimativen Tipps. , Geschlecht: Kein Witz: Auch Wassermelonen haben ein Geschlecht. Und...",berliner_kurier
1,Horror beim Wetter: Deutschland kippt auf Braun – Schockprognose!,N_A,"Der Sommer ist gekommen, um zu bleiben – und in den kommenden Tagen schaltet das Wetter den Turbo an. Die ersten Hitzetage des Jahres stehen uns bevor. Passend dazu gibt es neue Schock-Prognosen vom europäischen Wettermodell. Hier kommen die aktuellen Aussichten. , Es ist eine Rolle rückwärts, die krasser nicht ausfallen könnte. Vor vier Wochen hat das europäische Wettermodell noch mit einem eher nassen Juni gerechnet. Das US-Wettermodell CFS der Nationale Ozean- und Atmosphärenbehörde (NOAA...","Horror beim Wetter: Deutschland kippt auf Braun – Schockprognose! Der Sommer ist gekommen, um zu bleiben – und in den kommenden Tagen schaltet das Wetter den Turbo an. Die ersten Hitzetage des Jahres stehen uns bevor. Passend dazu gibt es neue Schock-Prognosen vom europäischen Wettermodell. Hier kommen die aktuellen Aussichten. , Es ist eine Rolle rückwärts, die krasser nicht ausfallen könnte. Vor vier Wochen hat das europäische Wettermodell noch mit einem eher nassen Juni gerechnet. Das US-...",berliner_kurier


## 2.3 CSV file

In [147]:
# Save df from 5 June to csv -> done
kurier_df.to_csv('kurier_5june.csv', index=False)

In [165]:
# Save df from 6 June to csv -> done
kurier_df.to_csv('kurier_6june.csv', index=False)

In [171]:
# Save df from 7 June to csv (not yet applied)
kurier_df.to_csv('kurier_7june.csv', index=False)